# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishigupgta1234-ux/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This finding ranks Average Position (43%), Impressions (32%), and Scroll Depth
(15%) as the top predictors of health score. My methodology question: health
score is explicitly defined in the paper as Impressions(30pts) + Position(30pts)
+ CTR(20pts) + Scroll(20pts) — so three of the four top "predictors" are
literally components of the target itself. The paper does flag this
("importance is descriptive rather than causal"), which I respect. My follow-up
question would be: since a holdout split protects against overfitting but not
against structural overlap between label and features, would swapping in a
genuinely independent target (e.g., clicks 30 days later) make this importance
ranking useful for actual optimization decisions, rather than mostly re-deriving
the scoring formula's own weights?

This model predicts growing vs. declining pages, where growth is defined from
a 30-day-vs-previous-30-day change. My methodology question: the paper doesn't
state whether the holdout split is time-aware or random. Some features (like
"days visible") could plausibly be computed over a window that overlaps the
same period used to define the label. Given what I learned building my own
Week-5 split — that a naive split can let information about the outcome window
leak into training — I'd ask: was the split time-aware (train on an earlier
period, test on a later one), or random? That distinction matters more here
than the accuracy number itself.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Before: random row-level split

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

url = "https://raw.githubusercontent.com/ishigupgta1234-ux/flyrank-ml-internship/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

df["label"] = (df["trend_direction"] == "down").astype(int)

features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "engaged_sessions_90d", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

# trying a normal random split here first, NOT grouped by client
# just to see what happens if I don't bother with the client grouping thing
train_naive, test_naive = train_test_split(df, test_size=0.20, random_state=42)

X_train = train_naive[features].fillna(0)
X_test = test_naive[features].fillna(0)
y_train = train_naive["label"]
y_test = test_naive["label"]

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

test_naive["model_score"] = model.predict_proba(X_test)[:, 1]

# same top-50 logic as week 5
top50_naive = test_naive.sort_values("model_score", ascending=False).head(50)
precision_naive = top50_naive["label"].mean()

print("naive split precision@50:", precision_naive)

# this is what i got in week 5 with the proper client split, just writing it here to compare
print("week 5 client-grouped precision@50: 0.68")

diff = precision_naive - 0.68
print("difference:", diff)

naive split precision@50: 0.92
week 5 client-grouped precision@50: 0.68
difference: 0.24


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit

# just checking i didn't accidentally leave the leaky columns in
leak_cols = ["trend_direction", "trend_pct"]

for col in leak_cols:
    if col in features:
        print(col, "-> FOUND IN FEATURES, this is a problem")
    else:
        print(col, "-> not in features, good")

trend_direction -> not in features, good
trend_pct -> not in features, good


The hard leakage check passes: trend_direction and trend_pct (the columns the
label is directly built from) were correctly excluded from the feature list in
Week 5 — confirmed above.

A softer risk worth naming: several features (impressions_90d, clicks_90d,
ctr, avg_position) are 90-day rolling aggregates computed over a window that
likely overlaps the same period the trend label is calculated from, rather
than being strictly "known before" the decision point. This isn't hard
leakage — the model can't see the label directly — but it means the honest
reading of Precision@50 = 0.68 is "the model separates decliners well within
this same-window snapshot," not "the model would have predicted decline in
advance." A stricter version of this lane would use only prior-period
features to predict a later-period label.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (from Week 5): "The Decision Tree beats the hand-written rule."

Rewritten: "In this client-level holdout, the decision tree ranked more true
decliners into its top 50 than the hand-written rule (0.68 vs 0.38) — observed
on one ~30k-row anonymized sample, one train/test split, and one label
definition. This is decision-support evidence that the tree outranks the rule
on this data, not a guarantee it would hold on a different month, client mix,
or sample."

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.